In [2]:
from crewai import Agent, Task, Crew, Process, LLM
from crewai.tools import BaseTool
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from dotenv import load_dotenv
from pydantic import Field
import os

load_dotenv()
os.environ["LITELLM_CACHE"] = "false"
os.environ["CREWAI_DISABLE_CACHE"] = "true"

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

# Use CrewAI's own LLM class
llm = LLM(
    model="groq/llama-3.3-70b-versatile",
    temperature=0.3
)

print("Imports successful")
print("LLM ready")

Imports successful
LLM ready


In [3]:
documents = [
    Document(page_content="Large Language Models (LLMs) are neural networks trained on massive text datasets. They predict the next token based on context. Popular LLMs include GPT-4, Claude, Gemini, and Llama.", metadata={"source": "ai_concepts"}),
    Document(page_content="RAG stands for Retrieval Augmented Generation. It combines a retrieval system with an LLM. The retrieval system fetches relevant documents, which are added to the LLM prompt as context to generate grounded answers.", metadata={"source": "ai_concepts"}),
    Document(page_content="Vector databases store text as numerical vectors called embeddings. They enable semantic search — finding documents by meaning rather than keywords. Popular vector databases include ChromaDB, Pinecone, and FAISS.", metadata={"source": "ai_concepts"}),
    Document(page_content="Prompt engineering is the practice of designing inputs to LLMs to get desired outputs. Key techniques include zero-shot, few-shot, chain-of-thought, and role prompting.", metadata={"source": "ai_concepts"}),
    Document(page_content="Multi-agent systems use multiple AI agents working together. Each agent has a specific role, goal, and set of tools. Frameworks like CrewAI, AutoGen, and Agno enable building multi-agent systems.", metadata={"source": "ai_concepts"}),
    Document(page_content="Hallucination in LLMs refers to generating confident but factually incorrect information. RAG reduces hallucination by grounding responses in retrieved factual documents.", metadata={"source": "ai_concepts"}),
    Document(page_content="Fine-tuning is the process of training a pre-trained LLM on a specific dataset to adapt it for a particular task or domain. LoRA and QLoRA are popular efficient fine-tuning techniques.", metadata={"source": "ai_concepts"}),
    Document(page_content="LangChain is a framework for building LLM applications. It provides abstractions for chains, retrievers, memory, and agents. It simplifies building RAG pipelines and agentic workflows.", metadata={"source": "ai_concepts"}),
    Document(page_content="Embeddings are numerical vector representations of text. Similar texts have similar embeddings. The all-MiniLM-L6-v2 model produces 384-dimensional embeddings for semantic search.", metadata={"source": "ai_concepts"}),
    Document(page_content="CrewAI is a multi-agent orchestration framework. It allows defining agents with roles, goals and tools. Agents collaborate through tasks in a crew to complete complex workflows.", metadata={"source": "ai_concepts"}),
]

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50
)

chunks = text_splitter.split_documents(documents)

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="ai_knowledge_base"
)

print(f"Documents loaded: {len(documents)}")
print(f"Chunks created: {len(chunks)}")
print(f"Vectorstore ready with {vectorstore._collection.count()} chunks")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Documents loaded: 10
Chunks created: 10
Vectorstore ready with 10 chunks


In [4]:
class RAGSearchTool(BaseTool):
    name: str = "search_knowledge_base"
    description: str = "Search the AI knowledge base to find relevant information about AI concepts, RAG, LLMs, embeddings, and multi-agent systems. Input should be a search query string."
    vectorstore: object = Field(default=None)

    def _run(self, query: str) -> str:
        retriever = self.vectorstore.as_retriever(
            search_kwargs={"k": 3}
        )
        docs = retriever.invoke(query)

        results = []
        for i,doc in enumerate(docs):
            results.append(f"Result {i+1}: {doc.page_content}")

        return "\n\n".join(results)


rag_tool = RAGSearchTool(vectorstore=vectorstore)

print("Testing RAG tool directly:")
print(rag_tool._run("What is RAG and how does it work?"))

Testing RAG tool directly:
Result 1: RAG stands for Retrieval Augmented Generation. It combines a retrieval system with an LLM. The retrieval system fetches relevant documents, which are added to the LLM prompt as context to generate grounded answers.

Result 2: Hallucination in LLMs refers to generating confident but factually incorrect information. RAG reduces hallucination by grounding responses in retrieved factual documents.

Result 3: LangChain is a framework for building LLM applications. It provides abstractions for chains, retrievers, memory, and agents. It simplifies building RAG pipelines and agentic workflows.


In [5]:
# AGENT 1 — Researcher
researcher = Agent(
    role="AI Knowledge Researcher",
    goal="Search the knowledge base thoroughly to find accurate and relevant information that answers the user's question completely.",
    backstory="""You are an expert AI researcher with deep knowledge of 
    machine learning, LLMs, RAG systems, and multi-agent frameworks. 
    You always search the knowledge base before answering and never 
    make up information. You retrieve evidence and present findings clearly.""",
    tools=[rag_tool],
    llm=llm,
    verbose=True,
    allow_delegation=False
)

# AGENT 2 — Summarizer
summarizer = Agent(
    role="AI Knowledge Summarizer",
    goal="Take the researcher's findings and produce a clear, concise, well-structured answer that directly addresses the user's question.",
    backstory="""You are an expert technical writer who specializes in 
    explaining complex AI concepts clearly. You receive research findings 
    and transform them into clean, readable answers. You never add 
    information not present in the research findings.""",
    tools=[],
    llm=llm,
    verbose=True,
    allow_delegation=False
)

print("Researcher agent ready")
print("Summarizer agent ready")

Researcher agent ready
Summarizer agent ready


In [6]:
research_task = Task(
    description="""Search the knowledge base to find comprehensive information 
    about the following question: {question}
    
    Use the search_knowledge_base tool to retrieve relevant information.
    Present all findings clearly with the retrieved evidence.""",
    expected_output="""A detailed research report containing:
    - All relevant information found in the knowledge base
    - Direct evidence from retrieved documents
    - Clear presentation of facts without any made-up information""",
    agent=researcher
)

# TASK 2 — Summarization task assigned to Summarizer agent
summarize_task = Task(
    description="""Take the research findings provided and create a clear, 
    concise answer to the original question: {question}
    
    Use ONLY the information from the research findings.
    Do not add any information not present in the research.""",
    expected_output="""A clean, well-structured answer that:
    - Directly answers the question
    - Is concise and easy to understand
    - Contains only information from the research findings
    - Is 3-5 sentences maximum""",
    agent=summarizer,
    context=[research_task]
)

print("Tasks defined successfully")
print("Task 1: Research — assigned to Researcher agent")
print("Task 2: Summarize — assigned to Summarizer agent")

Tasks defined successfully
Task 1: Research — assigned to Researcher agent
Task 2: Summarize — assigned to Summarizer agent


In [7]:
crew = Crew (
    agents=[researcher, summarizer],
    tasks=[research_task, summarize_task],
    process=Process.sequential,
    verbose=True
)

print("Crew assembled successfully")
print("Agents:", [agent.role for agent in crew.agents])
print("Tasks:", [task.description[:50] + "..." for task in crew.tasks])

Crew assembled successfully
Agents: ['AI Knowledge Researcher', 'AI Knowledge Summarizer']
Tasks: ['Search the knowledge base to find comprehensive in...', 'Take the research findings provided and create a c...']


In [8]:
import nest_asyncio
import asyncio
nest_asyncio.apply()

async def run_crew():
    result = await crew.kickoff_async(
        inputs={"question": "What is RAG and how does it reduce hallucination?"}
    )
    return result

result = asyncio.get_event_loop().run_until_complete(run_crew())

print("\n" + "="*60)
print("FINAL ANSWER:")
print("="*60)
print(result)

# Agent: AI Knowledge Researcher
## Task: Search the knowledge base to find comprehensive information 
    about the following question: What is RAG and how does it reduce hallucination?
    
    Use the search_knowledge_base tool to retrieve relevant information.
    Present all findings clearly with the retrieved evidence.


# Agent: AI Knowledge Researcher
## Thought: Thought: To answer the question about RAG and its role in reducing hallucination, I need to search the knowledge base for relevant information. The search query should include the terms "RAG" and "hallucination" to find the most accurate and comprehensive information.
## Using tool: search_knowledge_base
## Tool Input: 
"{\"query\": \"RAG and hallucination reduction in AI models\"}"
## Tool Output: 
Result 1: Hallucination in LLMs refers to generating confident but factually incorrect information. RAG reduces hallucination by grounding responses in retrieved factual documents.

Result 2: RAG stands for Retrieval Augmen